# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Deep Equilibrium Models (DEQ) & Implicit Differentiation
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Bai, S., Kolter, J. Z., & Koltun, V. (2019). *Deep Equilibrium Models*. NeurIPS.
- Bai, S., Koltun, V., & Kolter, J. Z. (2020). *Multiscale Deep Equilibrium Models*. NeurIPS.
- Winston, E., & Kolter, J. Z. (2020). *Monotone Operator Equilibrium Networks*. NeurIPS.
- Anderson, D. G. (1965). *Iterative Procedures for Nonlinear Integral Equations*. J. ACM.
- Griewank, A., & Walther, A. (2008). *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation*. SIAM.

## 2. Mathematical Foundations & Governing Equations

### Equilibrium Fixed-Point Formulation
Rather than stacking $L$ discrete layers $z_{k+1} = \sigma(W_k z_k + U_k x + b_k)$, a DEQ defines output
as the fixed point $z^\star$ of an implicit nonlinear equation:
$$z^\star = f_\theta(z^\star, x) = \tanh(z^\star W_z + x W_x + b).$$

### Banach Contraction Mapping Condition
Under $\|W_z\|_2 < 1$, the operator $f_\theta(\cdot, x)$ is a strict contraction with unique fixed point $z^\star$.

### Anderson Acceleration (Type II)
Solves a multi-secant least-squares problem over past residuals $\{R_j = f(z_j) - z_j\}$:
$$\min_\alpha \|\Delta R \alpha - R_k\|_2^2 \implies (\Delta R^T \Delta R + \epsilon I) \alpha = \Delta R^T R_k.$$

### Implicit Differentiation (IFT) & Adjoint Solve
By differentiating $z^\star - f_\theta(z^\star, x) = 0$, backward gradients require solving:
$$(I - J_f^T) v^\star = \nabla_{z^\star} \mathcal{L} \implies \frac{\mathrm{d}\mathcal{L}}{\mathrm{d}\theta} = (v^\star)^T \frac{\partial f_\theta(z^\star, x)}{\partial \theta}.$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import numpy as np

torch.manual_seed(42)
B, D, In_D = 4, 4, 3

raw_wz = torch.randn(D, D) * 0.3
# Enforce contractivity: spectral norm < 1
w_z = raw_wz / (torch.linalg.matrix_norm(raw_wz, ord=2) * 1.5)
w_x = torch.randn(In_D, D) * 0.4
b_z = torch.randn(D) * 0.1
x_input = torch.randn(B, In_D)

print(f"Input shape: {x_input.shape}, Latent D: {D}, ||W_z||_2: {torch.linalg.matrix_norm(w_z, ord=2):.4f}")

def f_theta(z, x, wz, wx, b):
    return torch.tanh(z @ wz + x @ wx + b)

In [ ]:
# 4. First-Principles Implementation & Native Parity
# 1. Picard Fixed-Point Iteration
z_picard = torch.zeros(B, D)
for _ in range(20):
    z_picard = f_theta(z_picard, x_input, w_z, w_x, b_z)
res_picard = z_picard - f_theta(z_picard, x_input, w_z, w_x, b_z)

# 2. Anderson Acceleration (Type II, m=2)
z_anderson = torch.zeros(B, D)
z_hist, g_hist = [], []
for k in range(15):
    g = f_theta(z_anderson, x_input, w_z, w_x, b_z)
    z_hist.append(z_anderson)
    g_hist.append(g)
    if len(z_hist) > 3:
        z_hist.pop(0)
        g_hist.pop(0)
    k_hist = len(z_hist) - 1
    if k_hist == 0:
        z_anderson = g
    else:
        z_next = torch.empty_like(z_anderson)
        for b_idx in range(B):
            R_mat = torch.stack([g_hist[j][b_idx] - z_hist[j][b_idx] for j in range(k_hist + 1)], dim=1)
            d_R = R_mat[:, 1:] - R_mat[:, :-1]
            reg = 1e-5 * torch.eye(k_hist)
            rhs = d_R.t() @ R_mat[:, -1:]
            lhs = d_R.t() @ d_R + reg
            alpha = torch.linalg.solve(lhs, rhs).squeeze(1)
            g_diff = torch.stack([g_hist[j+1][b_idx] - g_hist[j][b_idx] for j in range(k_hist)], dim=1)
            z_next[b_idx] = g_hist[-1][b_idx] - (g_diff @ alpha)
        z_anderson = z_next
res_anderson = z_anderson - f_theta(z_anderson, x_input, w_z, w_x, b_z)

# 3. Adjoint Linear Solve via Neumann Series: v_{t+1} = grad_z + J_f^T v_t
grad_loss_z = torch.randn(B, D) * 0.5
u = z_picard @ w_z + x_input @ w_x + b_z
d_act = 1.0 - torch.tanh(u)**2
adj_v = torch.zeros_like(grad_loss_z)
for _ in range(20):
    adj_v = grad_loss_z + (adj_v * d_act) @ w_z.t()

delta = adj_v * d_act
grad_x = delta @ w_x.t()
grad_wz = z_picard.t() @ delta

# 4. Monotone Operator Networks (MON)
m_mat = torch.randn(D, D) * 0.3
w_sym = m_mat.t() @ m_mat + 0.1 * torch.eye(D)
raw_skew = torch.randn(D, D) * 0.3
w_skew = raw_skew - raw_skew.t()
W_mon = w_sym + w_skew
w_mon_x = torch.randn(In_D, D) * 0.3
b_mon = torch.randn(D) * 0.1

z_mon = torch.zeros(B, D)
for _ in range(15):
    lin = z_mon @ W_mon + x_input @ w_mon_x + b_mon
    z_mon = torch.tanh(z_mon - 0.2 * lin)

print(f"Picard res: {torch.norm(res_picard):.2e}, Anderson res: {torch.norm(res_anderson):.2e}, Monotone norm: {torch.norm(z_mon):.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Residual convergence below threshold
assert torch.max(torch.abs(res_picard)).item() < 1e-4, "Picard failed to converge"
assert torch.max(torch.abs(res_anderson)).item() < 1e-4, "Anderson failed to converge"

# Invariant 2: Adjoint identity (I - J_f^T) v = grad_loss_z
# J_f^T v = (v * d_act) @ w_z.t()
lhs_adjoint = adj_v - (adj_v * d_act) @ w_z.t()
adjoint_err = torch.max(torch.abs(lhs_adjoint - grad_loss_z)).item()
assert adjoint_err < 1e-4, f"Adjoint linear system not satisfied: {adjoint_err:.2e}"

print(f"All quantitative invariants verified! Adjoint system residual error: {adjoint_err:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = f_theta(z_picard, x_input, w_z, w_x, b_z)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"DEQ cell evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Unconstrained weight matrix with spectral norm >> 1 causes explosive divergence
w_bad = torch.randn(D, D) * 2.5
z_div = torch.randn(B, D)
for _ in range(20):
    z_div = z_div @ w_bad + x_input @ w_x + b_z
print(f"Pathological unconstrained linear iteration norm: {torch.norm(z_div):.2e}")

# Corrected: Enforcing spectral normalization or monotone operator structuring guarantees bounded equilibrium
z_corr = torch.zeros(B, D)
for _ in range(20):
    z_corr = f_theta(z_corr, x_input, w_z, w_x, b_z)
print(f"Corrected bounded equilibrium norm: {torch.norm(z_corr):.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Infinite Depth with O(1) Memory**: Deep Equilibrium Models decouple network representation from backpropagation depth, computing exact gradients via the Implicit Function Theorem.
2. **Multi-Secant Acceleration**: Anderson acceleration dramatically shortens iteration counts over standard Picard iteration by solving inexpensive least-squares problems over recent residual history.
3. **Monotone Operators**: Structuring matrices into symmetric positive-definite and skew-symmetric components guarantees equilibrium existence across non-contractive dynamics.